<a href="https://colab.research.google.com/github/yogireddy02/Learning-GenAI/blob/main/AI_Feedback_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ⚙️ Day 4: Automated Feedback Analytics Pipeline
* **Core Concepts:** Batch processing using `for` loops, multi-stage file reading/writing.
* **Purpose:** Simulates an enterprise analytical system that processes customer reviews in bulk and compiles reports.
* **Status:** Complete ✅


In [ ]:
with open("customer_reviews.txt", "w") as f:
  f.write("1. The product delivery was delayed by 5 days. Very bad service.\n")
  f.write("2. Absolutely loved the quality! Best purchase of this year.\n")
  f.write("3. The app keeps crashing when I try to make a payment. Fix it.\n")

In [ ]:
import json
from google import genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

print("--- AI Feedback Pipeline Starting ---\n")

with open('customer_reviews.txt', 'r') as file:
  reviews = file.readlines()



positive_count = 0
negative_count = 0
extracted_issues = []

for review in reviews:
  if not review.strip():
    continue

  print(f'Analyzing: {review.strip()}')

  instructions = f"""
  Analyze this customer review. Extract two data points:
  1. 'sentiment': Must be exactly 'Positive' or 'Negative'.
  2. 'issue': A 3-word summary of the core complaint (or 'None' if positive).

  Return ONLY raw JSON formatting like this: {{"sentiment": "value", "issue": "value"}}
  Review: {review}
  """

  response = client.models.generate_content(
      model = 'gemini-2.5-flash',
      contents = instructions,
  )

  try:
    data = json.loads(response.text.strip())

    if data['sentiment'] == 'positive':
      positive_count += 1
    else:
      negative_count += 1
      extracted_issues.append(data['issue'])

  except Exception as e:
    print(f"Error reading AI output for this line: {e}")


print("\nGenerating final report...")
with open("pipeline_report.txt", 'w') as report:
  report.write("==== ENTERPRISE FEEDBACK REPORT ====\n")
  report.write(f"Total Reviews Processed: {positive_count}\n")
  report.write(f"Positive Sentiment: {positive_count}\n")
  report.write(f"Negative Sentiment: {negative_count}\n")
  report.write("\nKey Complaints Found:\n")
  for issue in extracted_issues:
    report.write(f"- {issue}\n")

print("[SUCCESS] 'pipeline_report.txt' has ben generated!")


--- AI Feedback Pipeline Starting ---

Analyzing: 1. The product delivery was delayed by 5 days. Very bad service.
Error reading AI output for this line: Expecting value: line 1 column 1 (char 0)
Analyzing: 2. Absolutely loved the quality! Best purchase of this year.
Error reading AI output for this line: Expecting value: line 1 column 1 (char 0)
Analyzing: 3. The app keeps crashing when I try to make a payment. Fix it.
Error reading AI output for this line: Expecting value: line 1 column 1 (char 0)

Generating final report...
[SUCCESS] 'pipeline_report.txt' has ben generated!
